# Laikipia farm digitisation: train on a Kaggle GPU

Trains the U-Net (background / farm interior / farm boundary) on chips cut on the laptop, lets you check it
visually and, optionally, scores it on held-out test blocks.

**Before running**
1. On the laptop, follow `RUN_STEPS.md` up to the chips, e.g. for Sentinel-2 experiment E1:
   `python -m farmmap tiles --config config_s2.yaml --set 'context.use=["dea"]' paths.tiles=data/tiles_s2_E1`,
   then zip the folder (`tiles_s2_E1.zip`). Context layers are already inside the chips as extra bands.
2. Kaggle → *Datasets → New dataset* → upload the zip, name it e.g. `laikipia-s2-tiles-e1`, **keep it private**.
3. This notebook: *Settings → Accelerator → GPU T4 or P100*, *Internet → On*, *Add input →* that dataset.
4. In **1. Settings** set `TILES_DATASET` to the dataset name and `CONFIG` to the config the chips were cut with
   (`config_s2.yaml` for Sentinel-2, `config.yaml` for Legion, `config_nicfi.yaml` for NICFI).
5. **Save Version → Save & Run All (commit)**: it runs in the background (up to 12 h) and keeps
   `/kaggle/working` (the model) as the notebook output.
6. To continue training later, add the previous version's output as an input; step 5 resumes from its `last.pt`.

One experiment per run: for E0 / E1 / E2, run this notebook three times with a different tiles dataset.

## 1. Settings

In [ ]:
from pathlib import Path

TILES_DATASET = "laikipia-s2-tiles-e1"       # name of the Kaggle dataset holding this experiment's chips
CONFIG = "config_s2.yaml"                    # config the chips were cut with: config_s2.yaml / config.yaml / config_nicfi.yaml
REPO = "https://github.com/mangaorphy/Farm_Digitisation.git"
EPOCHS = 40
ENCODER = "resnet34"                          # resnet50 / efficientnet-b4 are bigger; try after a first baseline
BATCH = 16                                    # lower to 8 if CUDA runs out of memory

WORK = Path("/kaggle/working")
MODELS = WORK / "models"

## 2. Install and get the code

In [ ]:
!pip install -q "segmentation-models-pytorch>=0.3.4" rasterio geopandas pyogrio
import subprocess, sys
code_dir = WORK / "Farm_Digitisation"
if not code_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(code_dir)], check=True)
PROJECT = code_dir / "laikipia-farm-mapper"
sys.path.insert(0, str(PROJECT))
import farmmap, torch
print("farmmap", farmmap.__version__, "| torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU accelerator!")

## 3. Find the tiles

In [ ]:
import pandas as pd
cands = [p.parent for p in Path("/kaggle/input").rglob("index.csv")]
assert cands, "No index.csv under /kaggle/input - add your tiles dataset as an input"
match = [c for c in cands if TILES_DATASET in str(c)]
assert match or len(cands) == 1, f"TILES_DATASET={TILES_DATASET!r} not found; inputs with chips: {cands}"
TILES = (match or cands)[0]
index = pd.read_csv(TILES / "index.csv")
print(TILES)
print((TILES / "meta.yaml").read_text())
print(index.split.value_counts())
print("mean labelled share %.2f, interior %.2f, boundary %.3f" % tuple(index[["labelled", "interior", "boundary"]].mean()))

A few chips with their masks (green = interior, red = boundary, grey = ignored):

In [ ]:
import numpy as np, rasterio, matplotlib.pyplot as plt

def show_rgb(img):
    return np.moveaxis(img[:3], 0, -1)

def overlay(mask):
    rgba = np.zeros(mask.shape + (4,))
    rgba[mask == 1] = (0, 1, 0, .35); rgba[mask == 2] = (1, 0, 0, .8); rgba[mask == 255] = (.5, .5, .5, .6)
    return rgba

sample = index.sample(min(6, len(index)), random_state=0)
fig, axes = plt.subplots(2, len(sample), figsize=(3.2 * len(sample), 6.4))
for k, tid in enumerate(sample.id):
    with rasterio.open(TILES / "images" / f"{tid}.tif") as s: img = s.read()
    with rasterio.open(TILES / "masks" / f"{tid}.tif") as s: m = s.read(1)
    axes[0, k].imshow(show_rgb(img)); axes[0, k].set_title(tid, fontsize=7)
    axes[1, k].imshow(show_rgb(img)); axes[1, k].imshow(overlay(m))
for a in axes.ravel(): a.axis("off")
plt.tight_layout()

## 4. Config for this run

In [ ]:
from farmmap import config
cfg = config.load(PROJECT / CONFIG, [  # CONFIG is set in 1. Settings
    f"paths.tiles={TILES}", f"paths.models={MODELS}", f"paths.outputs={WORK / 'outputs'}",
    f"train.epochs={EPOCHS}", f"train.batch_size={BATCH}", f"model.encoder={ENCODER}", "train.num_workers=4",
])
dict(cfg.model), dict(cfg.train)

## 5. Train (resumes automatically from a previous run's `last.pt` if attached)

In [ ]:
import shutil
MODELS.mkdir(parents=True, exist_ok=True)
prev = [p for p in Path("/kaggle/input").rglob("last.pt")]
if prev and not (MODELS / "last.pt").exists():
    print("Resuming from", prev[0])
    for f in prev[0].parent.glob("*"):
        shutil.copy(f, MODELS / f.name)

from farmmap.train import train
best = train(cfg)

## 6. Learning curves

In [ ]:
h = pd.read_csv(MODELS / "history.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
h.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax[0]); ax[0].set_title("loss")
h.plot(x="epoch", y=["iou_interior", "iou_boundary", "val_miou"], ax=ax[1]); ax[1].set_title("validation IoU")
ax[1].set_ylim(0, 1)
h.tail()

## 7. Look at predictions on validation/test chips (blocks the model never trained on)

In [ ]:
from farmmap import model as M
net, meta = M.load(MODELS / "best.pt", "cuda" if torch.cuda.is_available() else "cpu")
dev = next(net.parameters()).device
held = index[index.split.isin(["test", "val"])].sample(min(5, (index.split != "train").sum()), random_state=1)
fig, axes = plt.subplots(len(held), 3, figsize=(10, 3.4 * len(held)))
axes = np.atleast_2d(axes)
for k, tid in enumerate(held.id):
    with rasterio.open(TILES / "images" / f"{tid}.tif") as s: img = s.read()
    with rasterio.open(TILES / "masks" / f"{tid}.tif") as s: m = s.read(1)
    with torch.no_grad():
        p = net(torch.from_numpy(M.normalize(img))[None].to(dev)).softmax(1)[0].cpu().numpy()
    axes[k, 0].imshow(show_rgb(img)); axes[k, 0].set_title("image")
    axes[k, 1].imshow(show_rgb(img)); axes[k, 1].imshow(overlay(m)); axes[k, 1].set_title("your parcels")
    axes[k, 2].imshow(np.stack([p[2], p[1], np.zeros_like(p[0])], -1)); axes[k, 2].set_title("model: green=interior red=boundary")
for a in axes.ravel(): a.axis("off")
plt.tight_layout()

## 8. (Optional) Predict + score the held-out test blocks on the GPU

Faster than the laptop CPU. Upload the ingested scene(s) as a second private dataset (Sentinel-2:
`data/imagery/scenes_s2/*.tif`; Legion: `data/imagery/scenes/*.tif`) and set `SCENES_DATASET`.
If the model was trained with context layers, also upload `data/context/dea.tif` (and `gee.tif`) as a dataset
and set `CONTEXT_DATASET`. The parcels and working area come from the cloned repository, so no labels upload
is needed (set `LABELS_DATASET` only to use a different KMZ).

In [ ]:
SCENES_DATASET = ""     # e.g. "laikipia-s2-scenes"; leave empty to skip
LABELS_DATASET = ""     # optional: a dataset with other KMZ labels (default: the repo's data/labels/manual)
CONTEXT_DATASET = ""    # e.g. "laikipia-context" (dea.tif / gee.tif); only for models trained with context

if SCENES_DATASET:
    from farmmap import imagery, predict, polygonize, evaluate, export
    scenes = sorted(Path(f"/kaggle/input/{SCENES_DATASET}").rglob("*.tif"))
    mosaic = imagery.write_vrt(scenes, WORK / "mosaic.vrt")  # order = file name; for date order use scenes.json
    cfg.paths.mosaic = mosaic
    if LABELS_DATASET:
        cfg.paths.labels = Path(f"/kaggle/input/{LABELS_DATASET}")
    if CONTEXT_DATASET:  # predict re-attaches the context sources recorded in the model
        cfg.paths.context = next(Path(f"/kaggle/input/{CONTEXT_DATASET}").rglob("*.tif")).parent
    from farmmap.cli import _bounds
    import types
    bounds = _bounds(cfg, types.SimpleNamespace(split="test", bbox=None))
    prob = predict.predict(cfg, MODELS / "best.pt", mosaic, WORK / "outputs/probabilities_test.tif", bounds)
    farms = polygonize.polygonize(cfg, prob, WORK / "outputs/farms_test.gpkg")
    export.to_kmz(farms, WORK / "outputs/farms_test.kmz")
    result = evaluate.evaluate(cfg, WORK / "outputs/farms_test.gpkg", split="test")
    result

## 9. Download

From the notebook output (right panel, or the *Output* tab of the saved version) download
`models/best.pt` (+ `history.csv`) and, if step 8 ran, `outputs/`. On the laptop put `best.pt` in the folder of
its experiment, e.g. `data/models_s2_E1/best.pt`, then (see `RUN_STEPS.md` step 7):

    python -m farmmap run --config config_s2.yaml --split test --model data/models_s2_E1/best.pt --out outputs/s2/farms_E1.gpkg
    python -m farmmap evaluate --config config_s2.yaml --pred outputs/s2/farms_E1.gpkg